In [0]:
from pyspark.sql.functions import col, when, sum, round, to_timestamp, to_date, current_timestamp, date_format
from pyspark.sql.types import StringType

In [0]:
df_transacciones = spark.table("finanzas.bronze_transacciones")
df_clientes = spark.table("finanzas.bronze_clientes")

### Limipieza y Transformación de datos - Transacciones

In [0]:
df_transacciones = (
                    df_transacciones.withColumn("monto_PEN", when(col("moneda") == "PEN", col("monto")).otherwise(col("monto") * 3.6))
                                    .withColumn("fecha_hora_transaccion", to_timestamp("fecha_transaccion", "yyyy-MM-dd HH:mm:ss"))
                                    .withColumn("fecha_transaccion", col("fecha_hora_transaccion").cast("date"))
                                    .withColumn("hora_transaccion", date_format("fecha_hora_transaccion", "HH:mm:ss"))
                                    .withColumn("deposito_PEN", when(col("tipo_transaccion") == "DEPOSITO", col("monto_PEN")).otherwise(0))
                                    .withColumn("retiro_PEN", when(col("tipo_transaccion") == "RETIRO", col("monto_PEN")).otherwise(0))
                                    .withColumn("fecha_carga", current_timestamp())
                                    .select(
                                            "id_cliente",
                                            "fecha_transaccion",
                                            "hora_transaccion",
                                            "moneda", 
                                            "monto_PEN", 
                                            "deposito_PEN", 
                                            "retiro_PEN",
                                            "fecha_carga"
                                        )
                    )

### Limipieza y Transformación de datos - Clientes

In [0]:
def formato_documento(num_documento):
    if num_documento is None:
        return None
    
    documento = str(num_documento)

    if len(documento) <= 8:
        return ("000000" + documento)[-8:]
    else:
        return documento

In [0]:
udf_documento = udf(formato_documento, StringType())

In [0]:
df_clientes = (
                df_clientes.withColumn("documento", udf_documento("documento"))
                            .select(
                                    "id_cliente", "documento",
                                    col("representante").alias("nombre_completo")
                                    )
                            .withColumn("fecha_carga", current_timestamp())
            )

In [0]:
try:
    df_transacciones.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("finanzas.silver_transacciones")

    df_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("finanzas.silver_clientes")

except Exception as e:
    import traceback
    # Tipo de error
    error_type = type(e).__name__
    # Descripcion del error
    error_summary = str(e)
    # Traza del error (ver en que parte se generó el error)
    error_trace = traceback.format_exc()
    
    # Error completo
    error_msg_full = f"{error_type}: {error_summary}\n{error_trace}"

    if len(error_msg_full) > 500:
        error_msg = error_msg_full[:500] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full

    dbutils.jobs.taskValues.set(key="error", value=error_msg)
    raise e